# EMT scalar DC component validation

This notebook validates the Python bindings for:

- `dpsimpy.emt.dc.VoltageSource`
- `dpsimpy.emt.dc.CurrentSource`
- `dpsimpy.emt.dc.Resistor`
- `dpsimpy.emt.dc.Capacitor`
- `dpsimpy.emt.dc.Inductor`
- `dpsimpy.emt.dc.PiLine`

It runs five independent EMT/MNA cases, checks all logged values for `NaN`/`Inf`, compares final values with analytical references, and plots the transients. It **does not run CMake**. Build or install `dpsimpy` before executing the notebook.

Scalar DC sign convention used throughout:

- `v_intf = v_terminal1 - v_terminal0`
- positive `i_intf` flows from terminal 1 to terminal 0


In [ ]:
from __future__ import annotations

import importlib
import math
import sys
from pathlib import Path
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def import_dpsimpy():
    """Import an installed module or locate a locally built extension."""
    try:
        return importlib.import_module("dpsimpy")
    except ModuleNotFoundError as first_error:
        cwd = Path.cwd().resolve()
        repo_candidates = [cwd, *cwd.parents]
        extension_candidates: list[Path] = []
        for parent in repo_candidates:
            build_dir = parent / "build"
            if build_dir.exists():
                extension_candidates.extend(build_dir.rglob("dpsimpy*.so"))
        for extension in extension_candidates:
            sys.path.insert(0, str(extension.parent))
            try:
                return importlib.import_module("dpsimpy")
            except ModuleNotFoundError:
                sys.path.pop(0)
        raise ModuleNotFoundError(
            "dpsimpy is not importable. Build/install the Python module or "
            "start Jupyter with PYTHONPATH pointing to the directory that "
            "contains dpsimpy*.so."
        ) from first_error


dpsimpy = import_dpsimpy()
print("Loaded dpsimpy from:", Path(dpsimpy.__file__).resolve())

In [ ]:
LOG_ROOT = Path.cwd() / "logs" / "EMT_DC_Component_Validation"
LOG_ROOT.mkdir(parents=True, exist_ok=True)


def dc_node(name: str, initial_voltage: float = 0.0):
    node = dpsimpy.emt.SimNode(name, dpsimpy.PhaseType.DC)
    node.set_initial_voltage(complex(initial_voltage, 0.0))
    return node


def make_logger(case_name: str, entries: list[tuple[str, str, object]]):
    case_dir = LOG_ROOT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    dpsimpy.Logger.set_log_dir(str(case_dir))
    logger = dpsimpy.Logger(case_name)
    for log_name, attribute_name, component in entries:
        logger.log_attribute(log_name, attribute_name, component)
    return logger, case_dir / f"{case_name}.csv"


def run_simulation(
    case_name: str,
    system,
    logger,
    csv_path: Path,
    time_step: float,
    final_time: float,
    step_time: float | None = None,
    step_action: Callable[[], None] | None = None,
) -> pd.DataFrame:
    sim = dpsimpy.Simulation(case_name, dpsimpy.LogLevel.info)
    sim.set_domain(dpsimpy.Domain.EMT)
    sim.set_solver(dpsimpy.Solver.MNA)
    sim.set_system(system)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.add_logger(logger)

    sim.start()
    current_time = time_step  # Simulation::start() advances EMT time to dt.
    stepped = False
    try:
        while current_time <= final_time + 0.5 * time_step:
            if (
                step_action is not None
                and step_time is not None
                and not stepped
                and current_time >= step_time - 0.5 * time_step
            ):
                step_action()
                stepped = True
            current_time = sim.next()
    finally:
        sim.stop()

    if step_action is not None and not stepped:
        raise AssertionError(f"Step action was not executed in {case_name}.")
    if not csv_path.exists():
        raise FileNotFoundError(f"Expected DPsim log was not created: {csv_path}")

    frame = pd.read_csv(csv_path, skipinitialspace=True)
    frame.columns = [str(column).strip() for column in frame.columns]
    numeric = frame.select_dtypes(include=[np.number])
    if not np.isfinite(numeric.to_numpy()).all():
        bad_columns = numeric.columns[~np.isfinite(numeric.to_numpy()).all(axis=0)]
        raise AssertionError(
            f"{case_name} contains NaN or Inf in columns: {list(bad_columns)}"
        )
    return frame


def relative_error(actual: float, expected: float) -> float:
    return abs(actual - expected) / max(abs(expected), 1e-12)


def check_close(
    label: str,
    actual: float,
    expected: float,
    relative_tolerance: float,
    absolute_tolerance: float = 1e-9,
):
    error = relative_error(actual, expected)
    print(
        f"{label:28s} actual={actual:14.6f}  expected={expected:14.6f}  "
        f"relative error={error:.3e}"
    )
    if not math.isclose(
        actual,
        expected,
        rel_tol=relative_tolerance,
        abs_tol=absolute_tolerance,
    ):
        raise AssertionError(
            f"{label}: {actual} differs from {expected} beyond tolerance."
        )


def plot_series(
    frame: pd.DataFrame,
    columns: list[str],
    ylabel: str,
    title: str,
    step_time: float | None = None,
):
    plt.figure(figsize=(11, 4.5))
    for column in columns:
        plt.plot(frame["time"], frame[column], label=column)
    if step_time is not None:
        plt.axvline(step_time, linestyle="--", linewidth=1.0, label="step")
    plt.xlabel("Time [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 1. Voltage source and resistor

In [ ]:
case_name = "EMT_DC_01_VoltageSource_Resistor"
dt = 1e-5
final_time = 1e-3
voltage = 1_000.0
resistance = 10.0
expected_current = voltage / resistance

gnd = dpsimpy.emt.SimNode.gnd
node = dc_node("vs_r_node", voltage)
source = dpsimpy.emt.dc.VoltageSource("vs_r_source")
source.set_parameters(voltage)
source.connect([gnd, node])
load = dpsimpy.emt.dc.Resistor("vs_r_load")
load.set_parameters(resistance)
load.connect([gnd, node])

system = dpsimpy.SystemTopology(0.0, [gnd, node], [source, load])
logger, csv_path = make_logger(
    case_name,
    [
        ("v_node", "v", node),
        ("i_source", "i_intf", source),
        ("i_load", "i_intf", load),
        ("v_load", "v_intf", load),
    ],
)
vs_r = run_simulation(case_name, system, logger, csv_path, dt, final_time)

check_close("Voltage-source node", vs_r["v_node"].iloc[-1], voltage, 1e-8)
check_close("Resistor current", vs_r["i_load"].iloc[-1], expected_current, 1e-8)
check_close("Source current", vs_r["i_source"].iloc[-1], -expected_current, 1e-8)
vs_r["i_source_delivered"] = -vs_r["i_source"]
plot_series(vs_r, ["v_node", "v_load"], "Voltage [V]", "Voltage source + resistor")
plot_series(
    vs_r,
    ["i_source_delivered", "i_load"],
    "Current [A]",
    "Voltage source + resistor currents",
)

## 2. Current source and resistor

In [ ]:
case_name = "EMT_DC_02_CurrentSource_Resistor"
dt = 1e-5
final_time = 1e-3
current = 10.0
resistance = 50.0
expected_voltage = current * resistance

gnd = dpsimpy.emt.SimNode.gnd
node = dc_node("cs_r_node", expected_voltage)
source = dpsimpy.emt.dc.CurrentSource("cs_r_source")
source.set_parameters(current)
# terminal 1 is ground and terminal 0 is the node: positive source current is
# injected from ground into the DC node.
source.connect([node, gnd])
load = dpsimpy.emt.dc.Resistor("cs_r_load")
load.set_parameters(resistance)
load.connect([gnd, node])

system = dpsimpy.SystemTopology(0.0, [gnd, node], [source, load])
logger, csv_path = make_logger(
    case_name,
    [
        ("v_node", "v", node),
        ("i_source", "i_intf", source),
        ("v_source", "v_intf", source),
        ("i_load", "i_intf", load),
    ],
)
cs_r = run_simulation(case_name, system, logger, csv_path, dt, final_time)

check_close("Current-source node", cs_r["v_node"].iloc[-1], expected_voltage, 1e-8)
check_close("Resistor current", cs_r["i_load"].iloc[-1], current, 1e-8)
check_close("Current-source current", cs_r["i_source"].iloc[-1], current, 1e-8)
plot_series(cs_r, ["v_node"], "Voltage [V]", "Current source + resistor")
plot_series(
    cs_r, ["i_source", "i_load"], "Current [A]", "Current source + resistor currents"
)

## 3. Standalone inductor: RL voltage step

In [ ]:
case_name = "EMT_DC_03_RL_Step"
dt = 1e-5
step_time = 5e-3
final_time = 30e-3
stepped_voltage = 1_000.0
resistance = 5.0
inductance = 20e-3

gnd = dpsimpy.emt.SimNode.gnd
source_node = dc_node("rl_source_node", 0.0)
inductor_node = dc_node("rl_inductor_node", 0.0)
source = dpsimpy.emt.dc.VoltageSource("rl_source")
source.set_parameters(0.0)
source.connect([gnd, source_node])
resistor = dpsimpy.emt.dc.Resistor("rl_resistor")
resistor.set_parameters(resistance)
resistor.connect([inductor_node, source_node])
inductor = dpsimpy.emt.dc.Inductor("rl_inductor")
inductor.set_parameters(inductance, 0.0)
inductor.connect([gnd, inductor_node])

system = dpsimpy.SystemTopology(
    0.0, [gnd, source_node, inductor_node], [source, resistor, inductor]
)
logger, csv_path = make_logger(
    case_name,
    [
        ("v_source", "v", source_node),
        ("v_inductor_node", "v", inductor_node),
        ("i_resistor", "i_intf", resistor),
        ("i_inductor", "i_intf", inductor),
        ("v_inductor", "v_intf", inductor),
    ],
)
rl = run_simulation(
    case_name,
    system,
    logger,
    csv_path,
    dt,
    final_time,
    step_time,
    lambda: source.set_parameters(stepped_voltage),
)

elapsed = final_time - step_time
expected_current = (stepped_voltage / resistance) * (
    1.0 - math.exp(-resistance * elapsed / inductance)
)
check_close(
    "RL final inductor current", rl["i_inductor"].iloc[-1], expected_current, 5e-3
)
check_close(
    "RL current continuity", rl["i_resistor"].iloc[-1], rl["i_inductor"].iloc[-1], 1e-6
)
plot_series(
    rl, ["i_resistor", "i_inductor"], "Current [A]", "RL source step", step_time
)
plot_series(
    rl,
    ["v_source", "v_inductor_node", "v_inductor"],
    "Voltage [V]",
    "RL voltages",
    step_time,
)

## 4. Standalone capacitor: RC voltage step

In [ ]:
case_name = "EMT_DC_04_RC_Step"
dt = 1e-5
step_time = 5e-3
final_time = 60e-3
stepped_voltage = 1_000.0
resistance = 10.0
capacitance = 1e-3

gnd = dpsimpy.emt.SimNode.gnd
source_node = dc_node("rc_source_node", 0.0)
capacitor_node = dc_node("rc_capacitor_node", 0.0)
source = dpsimpy.emt.dc.VoltageSource("rc_source")
source.set_parameters(0.0)
source.connect([gnd, source_node])
resistor = dpsimpy.emt.dc.Resistor("rc_resistor")
resistor.set_parameters(resistance)
resistor.connect([capacitor_node, source_node])
capacitor = dpsimpy.emt.dc.Capacitor("rc_capacitor")
capacitor.set_parameters(capacitance)
capacitor.connect([gnd, capacitor_node])

system = dpsimpy.SystemTopology(
    0.0, [gnd, source_node, capacitor_node], [source, resistor, capacitor]
)
logger, csv_path = make_logger(
    case_name,
    [
        ("v_source", "v", source_node),
        ("v_capacitor", "v", capacitor_node),
        ("i_resistor", "i_intf", resistor),
        ("i_capacitor", "i_intf", capacitor),
        ("v_capacitor_intf", "v_intf", capacitor),
    ],
)
rc = run_simulation(
    case_name,
    system,
    logger,
    csv_path,
    dt,
    final_time,
    step_time,
    lambda: source.set_parameters(stepped_voltage),
)

elapsed = final_time - step_time
expected_voltage = stepped_voltage * (
    1.0 - math.exp(-elapsed / (resistance * capacitance))
)
check_close(
    "RC final capacitor voltage", rc["v_capacitor"].iloc[-1], expected_voltage, 5e-3
)
check_close(
    "RC interface voltage",
    rc["v_capacitor_intf"].iloc[-1],
    rc["v_capacitor"].iloc[-1],
    1e-6,
)
plot_series(
    rc,
    ["v_source", "v_capacitor", "v_capacitor_intf"],
    "Voltage [V]",
    "RC source step",
    step_time,
)
plot_series(rc, ["i_resistor", "i_capacitor"], "Current [A]", "RC currents", step_time)

## 5. Pi-line source-step validation

In [ ]:
case_name = "EMT_DC_05_PiLine_SourceStep"
dt = 1e-5
step_time = 0.05
final_time = 0.30
initial_source_voltage = 20e3
stepped_source_voltage = 22e3
feeder_resistance = 0.2
line_resistance = 0.5
line_inductance = 20e-3
total_line_capacitance = 2e-3
line_conductance = 0.0
load_resistance = 20.0

initial_current = initial_source_voltage / (
    feeder_resistance + line_resistance + load_resistance
)
initial_sending_voltage = initial_source_voltage - feeder_resistance * initial_current
initial_receiving_voltage = load_resistance * initial_current

final_current_expected = stepped_source_voltage / (
    feeder_resistance + line_resistance + load_resistance
)
final_sending_voltage_expected = (
    stepped_source_voltage - feeder_resistance * final_current_expected
)
final_receiving_voltage_expected = load_resistance * final_current_expected

gnd = dpsimpy.emt.SimNode.gnd
source_node = dc_node("pi_source_node", initial_source_voltage)
sending_node = dc_node("pi_sending_node", initial_sending_voltage)
receiving_node = dc_node("pi_receiving_node", initial_receiving_voltage)

source = dpsimpy.emt.dc.VoltageSource("pi_source")
source.set_parameters(initial_source_voltage)
source.connect([gnd, source_node])
feeder = dpsimpy.emt.dc.Resistor("pi_feeder")
feeder.set_parameters(feeder_resistance)
feeder.connect([sending_node, source_node])
line = dpsimpy.emt.dc.PiLine("pi_line")
line.set_parameters(
    line_resistance,
    line_inductance,
    total_line_capacitance,
    line_conductance,
    initial_current,
)
line.connect([receiving_node, sending_node])
load = dpsimpy.emt.dc.Resistor("pi_load")
load.set_parameters(load_resistance)
load.connect([gnd, receiving_node])

system = dpsimpy.SystemTopology(
    0.0,
    [gnd, source_node, sending_node, receiving_node],
    [source, feeder, line, load],
)
logger, csv_path = make_logger(
    case_name,
    [
        ("v_source_node", "v", source_node),
        ("v_sending_node", "v", sending_node),
        ("v_receiving_node", "v", receiving_node),
        ("i_source", "i_intf", source),
        ("i_feeder", "i_intf", feeder),
        ("i_line", "i_intf", line),
        ("i_load", "i_intf", load),
        ("v_line", "v_intf", line),
    ],
)
pi_line = run_simulation(
    case_name,
    system,
    logger,
    csv_path,
    dt,
    final_time,
    step_time,
    lambda: source.set_parameters(stepped_source_voltage),
)

check_close(
    "Pi-line final current", pi_line["i_line"].iloc[-1], final_current_expected, 1e-2
)
check_close(
    "Pi-line load current", pi_line["i_load"].iloc[-1], final_current_expected, 1e-2
)
check_close(
    "Pi-line feeder current", pi_line["i_feeder"].iloc[-1], final_current_expected, 1e-2
)
check_close(
    "Pi-line sending voltage",
    pi_line["v_sending_node"].iloc[-1],
    final_sending_voltage_expected,
    1e-2,
)
check_close(
    "Pi-line receiving voltage",
    pi_line["v_receiving_node"].iloc[-1],
    final_receiving_voltage_expected,
    1e-2,
)

plot_series(
    pi_line,
    ["v_source_node", "v_sending_node", "v_receiving_node"],
    "Voltage [V]",
    "DC Pi-line source step",
    step_time,
)
pi_line["i_source_delivered"] = -pi_line["i_source"]
plot_series(
    pi_line,
    ["i_source_delivered", "i_feeder", "i_line", "i_load"],
    "Current [A]",
    "DC Pi-line currents",
    step_time,
)

## Validation summary

In [ ]:
summary = pd.DataFrame(
    [
        {
            "case": "Voltage source + resistor",
            "metric": "load current [A]",
            "simulated": vs_r["i_load"].iloc[-1],
            "expected": 100.0,
        },
        {
            "case": "Current source + resistor",
            "metric": "node voltage [V]",
            "simulated": cs_r["v_node"].iloc[-1],
            "expected": 500.0,
        },
        {
            "case": "RL step",
            "metric": "inductor current [A]",
            "simulated": rl["i_inductor"].iloc[-1],
            "expected": (1_000.0 / 5.0)
            * (1.0 - math.exp(-5.0 * (0.030 - 0.005) / 20e-3)),
        },
        {
            "case": "RC step",
            "metric": "capacitor voltage [V]",
            "simulated": rc["v_capacitor"].iloc[-1],
            "expected": 1_000.0 * (1.0 - math.exp(-(0.060 - 0.005) / (10.0 * 1e-3))),
        },
        {
            "case": "Pi-line step",
            "metric": "line current [A]",
            "simulated": pi_line["i_line"].iloc[-1],
            "expected": final_current_expected,
        },
    ]
)
summary["relative_error"] = (
    summary["simulated"] - summary["expected"]
).abs() / summary["expected"].abs().clip(lower=1e-12)
summary

In [ ]:
assert np.isfinite(
    summary[["simulated", "expected", "relative_error"]].to_numpy()
).all()
print("All EMT scalar DC validation cases completed without NaN or Inf.")
print("Logs are stored in:", LOG_ROOT.resolve())